In [ ]:
import numpy as np
import random
from tensorflow import keras

In [25]:
with open('./dataset.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

print(f"Длина текста: {len(text)} символов")

Длина текста: 79761 символов


In [26]:
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

print(f"Уникальных символов: {len(chars)}")

Уникальных символов: 62


In [27]:
SEQ_LEN = 100
STEP = 3

sequences = []
next_chars = []

for i in range(0, len(text) - SEQ_LEN, STEP):
    sequences.append(text[i:i + SEQ_LEN])
    next_chars.append(text[i + SEQ_LEN])

print(f"Последовательностей: {len(sequences)}")

Последовательностей: 26554


In [28]:
X = np.zeros((len(sequences), SEQ_LEN, len(chars)), dtype=bool)
y = np.zeros((len(sequences), len(chars)), dtype=bool)

for i, seq in enumerate(sequences):
    for t, char in enumerate(seq):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

In [29]:
model = keras.Sequential([
    keras.layers.LSTM(128, input_shape=(SEQ_LEN, len(chars))),
    keras.layers.Dense(len(chars), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy')
model.summary()

C:\Users\Nick\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 128)            │        97,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,790 (413.24 KB)

 Trainable params: 105,790 (413.24 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = model.fit(X, y, batch_size=128, epochs=10)

Epoch 1/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 18s 82ms/step - loss: 3.2645
Epoch 2/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 18s 86ms/step - loss: 3.0219
Epoch 3/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 18s 84ms/step - loss: 2.7262
Epoch 4/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 16s 78ms/step - loss: 2.5894
Epoch 5/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - loss: 2.5141
Epoch 6/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 21s 99ms/step - loss: 2.4635
Epoch 7/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 19s 92ms/step - loss: 2.4302
Epoch 8/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 18s 87ms/step - loss: 2.3939
Epoch 9/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 18s 86ms/step - loss: 2.3657
Epoch 10/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 18s 85ms/step - loss: 2.3373


In [31]:
def generate_text(seed, length=500):
    result = seed
    for _ in range(length):
        x = np.zeros((1, SEQ_LEN, len(chars)))
        for t, char in enumerate(seed):
            x[0, t, char_to_idx[char]] = 1
        
        probs = model.predict(x, verbose=0)[0]
        next_idx = np.random.choice(len(chars), p=probs)
        next_char = idx_to_char[next_idx]
        
        result += next_char
        seed = seed[1:] + next_char
    return result

In [35]:
start = random.randint(0, len(text) - SEQ_LEN - 1)
seed = text[start:start + SEQ_LEN]

generated = generate_text(seed, 500)
print("РЕЗУЛЬТАТ ГЕНЕРАЦИИ:")
print(generated)

РЕЗУЛЬТАТ ГЕНЕРАЦИИ:
как ты думаешь, будет война?
— конечно, — уверенно сказал он. — зря, что ли, открыли столько училищ олана котося м гевыеву, пласах ухдавтий непола сольны ай. тейзниниха…
— фосиюу, прижешей прусетрииы салико о внейпи лебевяу. и рогом хамата скродбени, иглеслско.
— полероть, тво свеежнпу нахлайдо сказлели— — — тогру когоралькия поличто ны омувел: а — верункайще, впраго?
— о пзакролю встоуманостся поляеми сликия схорище зало веры. —  релкрежу тколанюя снакы ожо доняйвер уше…
дорапслизикару ог запилшыхи скоскте, снакантя. веслеэто дсогпел о нууент! кохороговарк? му закрокилк хсколномарнегой виежнк


In [36]:
with open('output.txt', 'w', encoding='utf-8') as f:
    f.write(generated)

model.save('trained_model.keras')
print("Обученная модель сохранена в 'trained_model.keras'")

Обученная модель сохранена в 'trained_model.keras'
